# CDN on BookCrossing - standalone notebook

## 0. Config and imports

In [ ]:
import copy
import itertools
import json
import re
import gc
from collections import Counter
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd

import torch
import torch._utils  # Keeps torch._dynamo imports stable in some notebook runtimes.
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


def set_seed(seed: int = 0):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


CFG = {
    # --------------------------------------------------------
    # Data
    # --------------------------------------------------------
    "work_dir": "../data",
    "min_user_interactions": 3,

    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------
    "embed_dim": 64,
    "tower_hidden": [256, 128, 64],
    "max_title_words": 2000,
    "max_authors": 200,
    "max_publishers": 200,
    "max_countries": 50,

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    "batch_size": 1024,
    "num_workers": 0,
    "grad_clip": 1.0,

    # --------------------------------------------------------
    # Grid search
    # --------------------------------------------------------
    "tune_fast": True,
    "tune_epochs": 3,
    "tune_val_fraction": 1.0,
    "skip_tune_if_cached": True,
    "cache_path": "bookcrossing_cdn_best_hparams.json",

    # --------------------------------------------------------
    # Final training
    # --------------------------------------------------------
    "final_epochs": 40,
    "seeds": [0, 1, 2, 3, 4],

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------
    "eval_k": [10, 50],
    "eval_batch_users": 128,
    "eval_item_batch": 1024,
    "head_fraction": 0.001,

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------
    "seed": 0,
}

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print(f"Final: {len(CFG['seeds'])} seeds × {CFG['final_epochs']} epochs")

## 1. Load BookCrossing

In [ ]:
def read_bookcrossing(raw: Path):
    users = pd.read_csv(
        raw / "BX-Users.csv",
        sep=";",
        encoding="latin-1",
        quotechar='"',
    )

    books = pd.read_csv(
        raw / "BX-Books.csv",
        sep=";",
        encoding="latin-1",
        quotechar='"',
        on_bad_lines="skip",
        low_memory=False,
    )
    # Drop image URL columns
    books = books.drop(columns=[c for c in books.columns if "Image-URL" in c], errors="ignore")
    # Clean year: non-numeric rows (e.g. publisher names) and out-of-range → 0
    year = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
    books["Year-Of-Publication"] = year.where(year.between(1800, 2024), 0).fillna(0).astype(int)

    ratings = pd.read_csv(
        raw / "BX-Book-Ratings.csv",
        sep=";",
        encoding="latin-1",
        quotechar='"',
    )
    # Keep only explicit ratings (implicit rating=0 is excluded)
    ratings = ratings[ratings["Book-Rating"] > 0].copy()

    return users, books, ratings


RAW_DIR = Path(CFG["work_dir"])
users_df, books_df, ratings = read_bookcrossing(RAW_DIR)

# Filter: ratings must reference books and users that exist in their tables
valid_isbns = set(books_df["ISBN"].tolist())
valid_uids = set(users_df["User-ID"].tolist())
ratings = ratings[ratings["ISBN"].isin(valid_isbns) & ratings["User-ID"].isin(valid_uids)].copy()

# Keep only users with enough interactions for leave-one-out
user_counts = ratings.groupby("User-ID").size()
active_users = user_counts[user_counts >= CFG["min_user_interactions"]].index
ratings = ratings[ratings["User-ID"].isin(active_users)].copy()

# Build index maps
all_user_ids = sorted(ratings["User-ID"].unique())
all_item_ids = sorted(ratings["ISBN"].unique())
uid_map = {u: i for i, u in enumerate(all_user_ids)}
iid_map = {b: i for i, b in enumerate(all_item_ids)}

NUM_USERS = len(uid_map)
NUM_ITEMS = len(iid_map)

print(f"users={NUM_USERS:,} items={NUM_ITEMS:,} ratings={len(ratings):,}")

## 2. Feature engineering

In [ ]:
def tokenize_title(title: str):
    if "(" in title:
        title = title.rsplit("(", 1)[0]
    return re.findall(r"[a-z0-9]+", title.lower())


def build_title_bow(titles: list, max_words: int) -> np.ndarray:
    counter = Counter()
    per_title = []
    for title in titles:
        toks = tokenize_title(str(title))
        per_title.append(toks)
        counter.update(toks)
    vocab = [w for w, _ in counter.most_common(max_words)]
    w2i = {w: i for i, w in enumerate(vocab)}
    mat = np.zeros((len(titles), len(vocab)), dtype=np.float32)
    for r, toks in enumerate(per_title):
        for tok in toks:
            j = w2i.get(tok)
            if j is not None:
                mat[r, j] = 1.0
    return mat


# Lookup dicts for fast per-ISBN access
isbn_to_title = books_df.set_index("ISBN")["Book-Title"].to_dict()
isbn_to_author = books_df.set_index("ISBN")["Book-Author"].to_dict()
isbn_to_year = books_df.set_index("ISBN")["Year-Of-Publication"].to_dict()
isbn_to_publisher = books_df.set_index("ISBN")["Publisher"].to_dict()

# ---------- item features ----------
titles = [str(isbn_to_title.get(isbn, "")) for isbn in all_item_ids]
authors = [str(isbn_to_author.get(isbn, "unknown")) for isbn in all_item_ids]
publishers = [str(isbn_to_publisher.get(isbn, "unknown")) for isbn in all_item_ids]
years_raw = np.array([float(isbn_to_year.get(isbn, 0)) for isbn in all_item_ids], dtype=np.float32)

# Title BOW
title_bow = build_title_bow(titles, CFG["max_title_words"])

# Author one-hot (top-N)
author_counter = Counter(authors)
top_author_map = {a: i for i, (a, _) in enumerate(author_counter.most_common(CFG["max_authors"]))}
author_mat = np.zeros((NUM_ITEMS, len(top_author_map)), dtype=np.float32)
for r, a in enumerate(authors):
    j = top_author_map.get(a)
    if j is not None:
        author_mat[r, j] = 1.0

# Publisher one-hot (top-N)
publisher_counter = Counter(publishers)
top_publisher_map = {p: i for i, (p, _) in enumerate(publisher_counter.most_common(CFG["max_publishers"]))}
publisher_mat = np.zeros((NUM_ITEMS, len(top_publisher_map)), dtype=np.float32)
for r, p in enumerate(publishers):
    j = top_publisher_map.get(p)
    if j is not None:
        publisher_mat[r, j] = 1.0

# Year normalized (items with unknown year=0 treated as a separate cluster)
years_norm = (years_raw - years_raw.mean()) / (years_raw.std() + 1e-6)

ITEM_GEN = np.hstack([
    author_mat,
    publisher_mat,
    years_norm.reshape(-1, 1),
    title_bow,
]).astype(np.float32)
ITEM_GEN = (ITEM_GEN - ITEM_GEN.mean(axis=0, keepdims=True)) / (ITEM_GEN.std(axis=0, keepdims=True) + 1e-6)
ITEM_GEN = ITEM_GEN.astype(np.float32)
ITEM_GEN_DIM = ITEM_GEN.shape[1]

# ---------- user features ----------
def age_bucket(age) -> int:
    """0=unknown, 1=<18, 2=18-24, 3=25-34, 4=35-44, 5=45-54, 6=55+"""
    try:
        a = float(age)
    except (TypeError, ValueError):
        return 0
    if a < 18:   return 1
    if a < 25:   return 2
    if a < 35:   return 3
    if a < 45:   return 4
    if a < 55:   return 5
    return 6


def extract_country(loc) -> str:
    if pd.isna(loc) or str(loc).strip() == "":
        return "unknown"
    parts = [p.strip() for p in str(loc).split(",")]
    return parts[-1].lower().strip() or "unknown"


users_df["age_bucket"] = users_df["Age"].apply(age_bucket)
users_df["country"] = users_df["Location"].apply(extract_country)

top_countries = set(users_df["country"].value_counts().head(CFG["max_countries"]).index)
users_df["country_key"] = users_df["country"].apply(lambda c: c if c in top_countries else "other")
country_map = {c: i for i, c in enumerate(sorted(users_df["country_key"].unique()))}

user_age_arr = np.zeros(NUM_USERS, dtype=np.int64)
user_country_arr = np.zeros(NUM_USERS, dtype=np.int64)
uid_set = set(uid_map.keys())
for _, row in users_df[users_df["User-ID"].isin(uid_set)].iterrows():
    u = uid_map[row["User-ID"]]
    user_age_arr[u] = row["age_bucket"]
    user_country_arr[u] = country_map.get(row["country_key"], 0)

USER_GEN = np.stack([user_age_arr, user_country_arr], axis=1).astype(np.int64)
# 7 age buckets (0-6), N country categories
USER_GEN_SIZES = [7, len(country_map)]

print(f"ITEM_GEN_DIM: {ITEM_GEN_DIM}")
print(f"USER_GEN_SIZES: {USER_GEN_SIZES}")

## 3. Leave-one-out split

In [ ]:
train_pairs = []
val_u, val_i = [], []
test_u, test_i = [], []

train_user_items = [set() for _ in range(NUM_USERS)]

for uid, grp in ratings.groupby("User-ID"):
    if uid not in uid_map:
        continue
    u = uid_map[uid]
    # No timestamp in BookCrossing: sort by ISBN string for a deterministic split
    item_seq = [iid_map[b] for b in sorted(grp["ISBN"].tolist()) if b in iid_map]
    if len(item_seq) < 3:
        continue
    for it in item_seq[:-2]:
        train_pairs.append((u, it))
        train_user_items[u].add(it)
    val_u.append(u)
    val_i.append(item_seq[-2])
    test_u.append(u)
    test_i.append(item_seq[-1])

train_pairs = np.asarray(train_pairs, dtype=np.int64)
val_u = np.asarray(val_u, dtype=np.int64)
val_i = np.asarray(val_i, dtype=np.int64)
test_u = np.asarray(test_u, dtype=np.int64)
test_i = np.asarray(test_i, dtype=np.int64)

print(f"train={len(train_pairs):,} val={len(val_u):,} test={len(test_u):,}")

known_val = [set(s) for s in train_user_items]

known_test = [set(s) for s in train_user_items]
for u, i in zip(val_u, val_i):
    known_test[int(u)].add(int(i))

item_freq = np.bincount(train_pairs[:, 1], minlength=NUM_ITEMS)

order = np.argsort(-item_freq)
n_head = max(1, int(NUM_ITEMS * CFG["head_fraction"]))
head_mask = np.zeros(NUM_ITEMS, dtype=bool)
head_mask[order[:n_head]] = True

print(f"head={head_mask.sum():,} tail={(~head_mask).sum():,}")
print(f"test_head_share={head_mask[test_i].mean():.4f}")
print(f"test_tail_share={(~head_mask[test_i]).mean():.4f}")

## 5. Dataset and CDN model

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs: np.ndarray):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return int(u), int(i)

class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: list[int], dropout: float = 0.0):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        self.net = nn.Sequential(*layers)
        self.out_dim = d
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class UserGenEnc(nn.Module):
    def __init__(self, sizes: list[int], dim: int = 16):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, dim) for n in sizes])
        self.out_dim = len(sizes) * dim
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([emb(x[:, i].long()) for i, emb in enumerate(self.embs)], dim=-1)

class CDN(nn.Module):
    def __init__(self, dropout: float = 0.0, n_experts: int = 2):
        super().__init__()
        h = CFG["tower_hidden"]
        ed = CFG["embed_dim"]
        self.user_id = nn.Embedding(NUM_USERS, ed)
        self.user_gen = UserGenEnc(USER_GEN_SIZES, dim=16)
        u_in = ed + self.user_gen.out_dim
        self.user_main = MLP(u_in, h, dropout)
        self.user_reg = MLP(u_in, h, dropout)
        self.user_ln = nn.LayerNorm(self.user_main.out_dim)
        self.item_id = nn.Embedding(NUM_ITEMS, ed)
        self.item_mem = MLP(ed, h, dropout)
        self.item_gen = MLP(ITEM_GEN_DIM, h, dropout)
        self.item_ln = nn.LayerNorm(self.item_mem.out_dim)
        self.gate = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
        self.init_weights()
    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def _user_input(self, u, ug):
        return torch.cat([self.user_id(u), self.user_gen(ug)], dim=-1)
    def user_vec_main(self, u, ug):
        return self.user_ln(self.user_main(self._user_input(u, ug)))
    def user_vec_reg(self, u, ug):
        return self.user_ln(self.user_reg(self._user_input(u, ug)))
    def user_vec(self, u, ug):
        return self.user_vec_main(u, ug)
    def item_vec(self, i, ig):
        mem = self.item_mem(self.item_id(i))
        gen = self.item_gen(ig)
        pop = torch.log1p(item_freq_t[i]).view(-1, 1)
        g = self.gate(pop)
        return self.item_ln(g * mem + (1.0 - g) * gen)
    def cdn_losses(self, main_users, main_items, reg_users, reg_items, alpha: float):
        loss_m = inbatch_softmax_loss(self.user_vec_main(main_users, ug_t[main_users]), self.item_vec(main_items, ig_t[main_items]))
        loss_r = inbatch_softmax_loss(self.user_vec_reg(reg_users, ug_t[reg_users]), self.item_vec(reg_items, ig_t[reg_items]))
        return alpha * loss_m + (1.0 - alpha) * loss_r, loss_m, loss_r

def inbatch_softmax_loss(user_vecs, item_vecs):
    u = F.normalize(user_vecs, dim=-1, eps=1e-6)
    v = F.normalize(item_vecs, dim=-1, eps=1e-6)
    logits = torch.clamp(u @ v.T, -20.0, 20.0)
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)

ug_t = torch.from_numpy(USER_GEN).long().to(device)
ig_t = torch.from_numpy(ITEM_GEN).float().to(device)
item_freq_t = torch.from_numpy(item_freq.astype(np.float32)).to(device)

def make_loader(pairs, batch_size=None, shuffle=True, drop_last=True):
    return DataLoader(PairDataset(pairs), batch_size=batch_size or CFG["batch_size"], shuffle=shuffle, drop_last=drop_last, num_workers=CFG["num_workers"], pin_memory=torch.cuda.is_available())

train_loader = make_loader(train_pairs)
print("CDN helpers ready")


## 4. Rebalanced distribution Omega_r

In [ ]:
def build_rebalanced_pairs(
    pairs: np.ndarray,
    head_mask: np.ndarray,
    seed: int = 0,
    keep_head_ratio: float = 0.35,
) -> np.ndarray:
    """Omega_r: downsample head interactions and keep all tail interactions."""
    rng = np.random.default_rng(seed)
    head_pairs = pairs[head_mask[pairs[:, 1]]]
    tail_pairs = pairs[~head_mask[pairs[:, 1]]]

    n_head_keep = min(len(head_pairs), max(1, int(len(head_pairs) * keep_head_ratio)))
    if len(head_pairs) > n_head_keep:
        head_pairs = head_pairs[rng.choice(len(head_pairs), n_head_keep, replace=False)]

    out = np.vstack([head_pairs, tail_pairs]) if len(head_pairs) else tail_pairs.copy()
    rng.shuffle(out)
    return out.astype(np.int64)


rebalanced_pairs = build_rebalanced_pairs(train_pairs, head_mask, seed=CFG["seed"])
rebalanced_loader = make_loader(rebalanced_pairs)

print(f"Omega_r rebalanced pairs: {len(rebalanced_pairs):,}")
print(f"Omega_r / Omega_m: {len(rebalanced_pairs) / max(len(train_pairs), 1):.3f}")


## 6. Evaluation

In [ ]:
@torch.no_grad()
def evaluate_full_corpus(
    model: nn.Module,
    users: np.ndarray,
    true_items: np.ndarray,
    known_user_items: List[set],
    head_mask: np.ndarray,
    ks: List[int],
    item_freq: np.ndarray,
    user_batch_size: int = 128,
    item_batch_size: int = 1024,
):
    model.eval()
    ranks_all, ranks_head, ranks_tail = [], [], []
    max_k = max(ks)
    recommended_by_k = {k: [] for k in ks}

    item_vectors = []
    for s in range(0, NUM_ITEMS, item_batch_size):
        e = min(s + item_batch_size, NUM_ITEMS)
        idx = torch.arange(s, e, device=device)
        v = model.item_vec(idx, ig_t[idx])
        v = F.normalize(v, dim=-1, eps=1e-6)
        if not torch.isfinite(v).all():
            raise RuntimeError(f"Non-finite item vectors on batch {s}:{e}")
        item_vectors.append(v.cpu())
    item_vectors = torch.cat(item_vectors, dim=0).to(device)

    for start in range(0, len(users), user_batch_size):
        end = min(start + user_batch_size, len(users))
        bu = users[start:end]
        bi = true_items[start:end]
        ut = torch.tensor(bu, device=device, dtype=torch.long)
        uvec = model.user_vec(ut, ug_t[ut])
        uvec = F.normalize(uvec, dim=-1, eps=1e-6)
        if not torch.isfinite(uvec).all():
            raise RuntimeError(f"Non-finite user vectors on batch {start}:{end}")

        scores = (uvec @ item_vectors.T).cpu().numpy()
        if not np.isfinite(scores).all():
            bad = np.sum(~np.isfinite(scores))
            raise RuntimeError(f"Non-finite scores in user batch {start}:{end}: {bad}")

        for row, (u, true_i) in enumerate(zip(bu, bi)):
            u = int(u)
            true_i = int(true_i)
            srow = scores[row].copy()
            for it in known_user_items[u]:
                if it != true_i:
                    srow[it] = -1e9
            true_score = srow[true_i]
            eps = 1e-12
            num_greater = int((srow > true_score + eps).sum())
            num_tied = int((np.abs(srow - true_score) <= eps).sum())
            rank = num_greater + num_tied - 1
            ranks_all.append(rank)
            if head_mask[true_i]:
                ranks_head.append(rank)
            else:
                ranks_tail.append(rank)
            if max_k < len(srow):
                top_unsorted = np.argpartition(-srow, max_k - 1)[:max_k]
                top_idx = top_unsorted[np.argsort(-srow[top_unsorted])]
            else:
                top_idx = np.argsort(-srow)
            for k in ks:
                recommended_by_k[k].append(top_idx[:k].astype(np.int64))

    def agg_accuracy(ranks: List[int]) -> Dict[str, float]:
        a = np.asarray(ranks, dtype=np.int64)
        out = {}
        for k in ks:
            if len(a) == 0:
                out[f"HR@{k}"] = np.nan
                out[f"NDCG@{k}"] = np.nan
            else:
                hits = a < k
                out[f"HR@{k}"] = 100.0 * hits.mean()
                out[f"NDCG@{k}"] = 100.0 * np.where(hits, 1.0 / np.log2(a + 2), 0.0).mean()
        return out

    tail_mask = ~head_mask
    num_tail_items = int(tail_mask.sum())
    popularity = np.log1p(item_freq.astype(np.float64))
    long_tail = {}

    for k in ks:
        recs = np.vstack(recommended_by_k[k])
        unique_recs = np.unique(recs)
        catalog_coverage = len(unique_recs) / NUM_ITEMS
        tail_coverage = np.sum(tail_mask[unique_recs]) / num_tail_items if num_tail_items > 0 else np.nan
        avg_popularity = popularity[recs].mean()
        tail_ratio = tail_mask[recs].mean()
        n_users_eval = recs.shape[0]
        if n_users_eval <= 1:
            personalization = np.nan
        else:
            exposure_counts = np.bincount(recs.reshape(-1), minlength=NUM_ITEMS)
            pairwise_intersections = np.sum(exposure_counts * (exposure_counts - 1) / 2)
            num_user_pairs = n_users_eval * (n_users_eval - 1) / 2
            avg_overlap = pairwise_intersections / num_user_pairs
            personalization = 1.0 - avg_overlap / k
        long_tail[f"CatalogCoverage@{k}"] = 100.0 * catalog_coverage
        long_tail[f"TailCoverage@{k}"] = 100.0 * tail_coverage
        long_tail[f"AvgPopularity@{k}"] = float(avg_popularity)
        long_tail[f"TailRatio@{k}"] = 100.0 * tail_ratio
        long_tail[f"Personalization@{k}"] = 100.0 * personalization

    return {
        "overall": agg_accuracy(ranks_all),
        "head": agg_accuracy(ranks_head),
        "tail": agg_accuracy(ranks_tail),
        "long_tail": long_tail,
        "counts": {
            "overall": len(ranks_all),
            "head": len(ranks_head),
            "tail": len(ranks_tail),
        },
    }


def print_metrics(metrics: Dict):
    print("counts:", metrics.get("counts", {}))
    for split in ("overall", "head", "tail"):
        print(f"[{split}]", metrics[split])
    if "long_tail" in metrics:
        print("[long_tail]", metrics["long_tail"])

## 7. CDN training helpers

In [ ]:
def alpha_for_epoch(epoch: int, total_epochs: int, gamma: float) -> float:
    return float(max(0.0, min(1.0, 1.0 - (epoch / (gamma * total_epochs)) ** 2)))


def train_cdn(model, optimizer, epochs, gamma, main_loader, reg_loader, val_u_eval, val_i_eval, seed_tag: str, track_val=True):
    best_hr = -1.0; best_state = None; logs = []
    for ep in range(1, epochs + 1):
        model.train(); alpha = alpha_for_epoch(ep, epochs, gamma); total_loss = 0.0; total_n = 0; reg_iter = itertools.cycle(reg_loader)
        pbar = tqdm(main_loader, desc=f"CDN {seed_tag} {ep}/{epochs}", leave=False)
        for main_users, main_items in pbar:
            reg_users, reg_items = next(reg_iter)
            main_users = main_users.to(device, non_blocking=True); main_items = main_items.to(device, non_blocking=True)
            reg_users = reg_users.to(device, non_blocking=True); reg_items = reg_items.to(device, non_blocking=True)
            loss, loss_m, loss_r = model.cdn_losses(main_users, main_items, reg_users, reg_items, alpha)
            if not torch.isfinite(loss): raise RuntimeError(f"Non-finite CDN loss: {loss.item()}")
            optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"]); optimizer.step()
            bs = main_users.size(0); total_loss += loss.item() * bs; total_n += bs; pbar.set_postfix(loss=f"{loss.item():.4f}", alpha=f"{alpha:.3f}")
        avg_loss = total_loss / max(total_n, 1); log = {"epoch": ep, "alpha": alpha, "train_loss": avg_loss}
        if track_val:
            met = evaluate_full_corpus(model, val_u_eval, val_i_eval, known_val, head_mask, CFG["eval_k"], item_freq, CFG["eval_batch_users"], CFG["eval_item_batch"])
            hr = met["overall"].get("HR@50", -1.0); log["val_HR@50"] = hr
            if hr > best_hr: best_hr = hr; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        logs.append(log)
    return best_state, best_hr, logs


## 8. Grid search

In [ ]:
LR_GRID = [0.001]
DROPOUT_GRID = [0.1]
WD_GRID = [0.0]
GAMMA_GRID = [2.0]
EXPERT_COUNT_GRID = [2]
combos = list(itertools.product(LR_GRID, DROPOUT_GRID, WD_GRID, GAMMA_GRID, EXPERT_COUNT_GRID))
cache_path = Path(CFG["cache_path"]); leaderboard_csv_path = cache_path.with_suffix(".leaderboard.csv")
if CFG["skip_tune_if_cached"] and cache_path.exists():
    best_hp = json.loads(cache_path.read_text()); leaderboard_df = pd.read_csv(leaderboard_csv_path) if leaderboard_csv_path.exists() else None
else:
    tune_ep = CFG["tune_epochs"] if CFG["tune_fast"] else CFG["final_epochs"]
    best_hr = -1.0; best_hp = None; leaderboard = []
    for ti, (lr, dr, wd, gamma, n_experts) in enumerate(combos, 1):
        set_seed(CFG["seed"]); model = CDN(dropout=dr, n_experts=n_experts).to(device); optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        status, error_message, hr = "ok", "", -1.0
        try:
            _, hr, _ = train_cdn(model, optimizer, tune_ep, gamma, train_loader, rebalanced_loader, val_u, val_i, f"trial{ti}", track_val=True)
        except RuntimeError as e:
            status, error_message = "failed", str(e)
        leaderboard.append({"trial": ti, "status": status, "error": error_message, "lr": lr, "dropout": dr, "weight_decay": wd, "gamma": gamma, "n_experts": n_experts, "val_HR@50": hr})
        if status == "ok" and hr > best_hr:
            best_hr = hr; best_hp = {"lr": lr, "dropout": dr, "weight_decay": wd, "gamma": gamma, "n_experts": n_experts, "val_HR@50": hr}
        del model, optimizer; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    leaderboard_df = pd.DataFrame(leaderboard).sort_values("val_HR@50", ascending=False, na_position="last")
    leaderboard_df.to_csv(leaderboard_csv_path, index=False)
    if best_hp is None: raise RuntimeError("No successful grid-search trial.")
    cache_path.write_text(json.dumps(best_hp, indent=2))
leaderboard_df.head(10) if leaderboard_df is not None else None


## 9. Final training over 5 seeds

In [ ]:
all_rows = []
all_test = []
print("Using best_hp:", best_hp)
for seed in CFG["seeds"]:
    print("\n" + "=" * 80); print(f"BookCrossing CDN seed {seed}"); print("=" * 80)
    set_seed(seed)
    rb_pairs = build_rebalanced_pairs(train_pairs, head_mask, seed=seed)
    rb_loader = make_loader(rb_pairs)
    model = CDN(dropout=best_hp["dropout"], n_experts=int(best_hp.get("n_experts", 2))).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=best_hp["lr"], weight_decay=best_hp["weight_decay"])
    best_state, best_val_hr50, logs = train_cdn(model, optimizer, CFG["final_epochs"], float(best_hp["gamma"]), train_loader, rb_loader, val_u, val_i, f"seed{seed}", track_val=True)
    assert best_state is not None
    model.load_state_dict(best_state); model.to(device); model.eval()
    test_metrics = evaluate_full_corpus(model, test_u, test_i, known_test, head_mask, CFG["eval_k"], item_freq, CFG["eval_batch_users"], CFG["eval_item_batch"])
    print("TEST METRICS"); print_metrics(test_metrics); all_test.append(test_metrics)
    row = {"method": "CDN", "seed": seed, "lr": best_hp["lr"], "dropout": best_hp["dropout"], "weight_decay": best_hp["weight_decay"], "gamma": best_hp["gamma"], "n_experts": best_hp.get("n_experts", 2), "best_val_HR@50": best_val_hr50}
    for split in ("overall", "head", "tail"):
        for key, value in test_metrics[split].items(): row[f"test_{split}_{key}"] = value
    for key, value in test_metrics.get("long_tail", {}).items(): row[f"test_{key}"] = value
    for key, value in test_metrics.get("counts", {}).items(): row[f"test_count_{key}"] = value
    all_rows.append(row)
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
metrics_df = pd.DataFrame(all_rows)
metrics_df


## 10. Compact final summary

In [ ]:
def make_bookcrossing_summary_table(
    metrics_df: pd.DataFrame,
    method_name: str = "CDN",
    save_path: str | None = "bookcrossing_cdn_summary.csv",
) -> pd.DataFrame:
    """One final table: mean ± std over seeds."""
    selected_metrics = [
        "test_overall_HR@10",
        "test_overall_NDCG@10",
        "test_overall_HR@50",
        "test_overall_NDCG@50",
        "test_head_HR@50",
        "test_head_NDCG@50",
        "test_tail_HR@50",
        "test_tail_NDCG@50",
        "test_CatalogCoverage@50",
        "test_TailCoverage@50",
        "test_AvgPopularity@50",
        "test_TailRatio@50",
        "test_Personalization@50",
    ]

    row = {
        "method": method_name,
        "num_seeds": metrics_df["seed"].nunique() if "seed" in metrics_df.columns else len(metrics_df),
        "head_fraction": CFG["head_fraction"],
        "head_share": f"{100 * CFG['head_fraction']:.1f}%",
        "num_head_items": int(head_mask.sum()),
        "num_tail_items": int((~head_mask).sum()),
        "test_head_share": f"{100 * float(head_mask[test_i].mean()):.2f}%",
        "test_tail_share": f"{100 * float((~head_mask[test_i]).mean()):.2f}%",
    }

    for hp_col in ["lr", "dropout", "weight_decay", "beta", "gamma", "gamma_0", "gamma_1", "n_experts", "final_epochs"]:
        if hp_col in metrics_df.columns:
            vals = metrics_df[hp_col].dropna().unique()
            if len(vals) == 1:
                row[hp_col] = vals[0]

    if "best_val_HR@50" in metrics_df.columns:
        vals = metrics_df["best_val_HR@50"].dropna().to_numpy(dtype=float)
        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        row["best_val_HR@50"] = f"{mean:.2f} ± {std:.2f}"

    for metric in selected_metrics:
        if metric not in metrics_df.columns:
            continue
        vals = metrics_df[metric].dropna().to_numpy(dtype=float)
        out_name = metric.replace("test_", "")
        if len(vals) == 0:
            row[out_name] = "nan"
            continue
        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        if "AvgPopularity" in out_name:
            row[out_name] = f"{mean:.4f} ± {std:.4f}"
        else:
            row[out_name] = f"{mean:.2f} ± {std:.2f}"

    summary_df = pd.DataFrame([row])
    if save_path is not None:
        summary_df.to_csv(save_path, index=False)
        print(f"saved: {save_path}")
    return summary_df


final_summary_df = make_bookcrossing_summary_table(
    metrics_df=metrics_df,
    method_name="CDN",
    save_path="bookcrossing_cdn_summary.csv",
)
final_summary_df